In [1]:
import onnxruntime as ort 

import pandas as pd
import numpy as np
import json

import matplotlib.pyplot as plt
%matplotlib inline

import plotly.express as px

In [2]:
def mark_anomaly(x, y, t, lower_limit, upper_limit):
    """
    Marks anomaly based on absolute reconstruction error 
    of the center timestamp load value.

    Parameters:
        x : np.ndarray  -> input tensor (batch, seq_len, features)
        y : np.ndarray  -> reconstructed/output tensor (batch, seq_len, features)
        t : int         -> center timestamp index
        lower_limit : float -> lower error threshold
        upper_limit : float -> upper error threshold

    Returns:
        anomalies : np.ndarray (bool) -> True where anomaly detected
    """

    # Error at center timestamp for load feature (index 0)
    error = x[:, t, 0] - y[:, t, 0]

    # Mark anomaly if error is outside control band
    anomalies = (error < lower_limit) | (error > upper_limit)

    return anomalies

In [28]:
def display_prediction(X_input, preds, t, lower_limit, upper_limit, anomalies=None):
    
    # Get anomalies: 
    if anomalies is not None:
        anomaly = anomalies
    else:
        anomaly = mark_anomaly(X_input, preds, t, lower_limit, upper_limit)

    # Create DataFrames for actual and predicted values
    df1 = pd.DataFrame({'x': range(len(X_input)), 'y': X_input[:, t, 0], 'color': 'Actual', 'anomaly': anomaly})
    df2 = pd.DataFrame({'x': range(len(preds)), 'y': preds[:, t, 0], 'color': 'Predicted', 'anomaly': False})
    
    # Concatenate the DataFrames
    df = pd.concat([df1, df2])
    fig = px.line(df, x='x', y='y', color='color')

    df_anomaly = df[df['anomaly']==True]
    print("Number of anomalies: ", len(df_anomaly))
    scatter_data = px.scatter(df_anomaly, x='x', y='y', color_discrete_sequence=['black']).data[0]
    scatter_data.update(marker=dict(size=5))
    fig.add_trace(scatter_data)

    fig.show()


In [5]:
# Get the upper and lower limits from the metadata file: 
def load_thresholds(json_path: str):
    """
    Load anomaly thresholds from JSON configuration file.
    """
    with open(json_path, "r") as f:
        config = json.load(f)

    upper_limit = config["upper_limit"]
    lower_limit = config["lower_limit"]

    return lower_limit, upper_limit

lower_limit, upper_limit = load_thresholds("D:/baseline_improvement-main/src/deployment_work/artifacts/latest/metadata.json")

In [6]:
# Load test data: 
X_inputs = np.load("../artifacts/latest/test_dataset.npy") 
X_inputs.shape

(1257, 10, 5)

In [13]:
# Convert to float dtype: 
input_for_model = X_inputs.astype(np.float32)

In [14]:
# ===== Run inference on model ======

# Start the onnx runtime session: 
session = ort.InferenceSession(
    "../artifacts/latest/models/model.onnx",
    providers=["CPUExecutionProvider"]
)

# get the input_name expected by onnx for input: 
input_name = session.get_inputs()[0].name  # this is "input_sequence" 

# Pass input to onnx session and run the prediction
pred_onnx = session.run(None,{input_name: input_for_model})[0]

pred_onnx.shape

(1257, 10, 5)

In [16]:
# Mark anomalies and plot the baseline for the 5-th timestamp in the series of 10 points recieved: 
anomaly = mark_anomaly(input_for_model, pred_onnx, 4, lower_limit, upper_limit)

In [17]:
display_prediction(input_for_model, pred_onnx, 4, lower_limit, upper_limit)

Number of anomalies:  147


In [ ]:
# ===== Scale back all feature values to the original range =====
#       (inverse transform for full feature set)

def minmax_inverse_3d(data_3d, min_vals, max_vals):
    """
    Inverse MinMax scaling for 3D data.
    Returns data in the SAME format as received.

    Parameters
    ----------
    data_3d : list or np.ndarray
        Shape = (samples, timesteps, features)
    min_vals : list
        Per-feature minimum values
    max_vals : list
        Per-feature maximum values

    Returns
    -------
    Same type as input (list or np.ndarray)
        Inverse transformed 3D data
    """

    # Detect original type
    is_numpy = isinstance(data_3d, np.ndarray)

    # Convert to list for safe processing
    working_data = data_3d.tolist() if is_numpy else data_3d

    original_data = []

    for sample in working_data:
        new_sample = []

        for row in sample:
            new_row = []

            for i in range(len(row)):
                x_norm = row[i]
                min_v = min_vals[i]
                max_v = max_vals[i]

                if max_v == min_v:
                    value = min_v
                else:
                    value = x_norm * (max_v - min_v) + min_v

                new_row.append(value)

            new_sample.append(new_row)

        original_data.append(new_sample)

    # Return in original format
    if is_numpy:
        return np.array(original_data)
    else:
        return original_data

In [ ]:
def minmax_inverse_load_only_3d(data_3d, min_vals, max_vals):
    """
    Inverse MinMax scaling ONLY for first feature (load).
    Returns data in the SAME format as received.
    """

    load_min = min_vals[0]
    load_max = max_vals[0]

    # Detect original type
    is_numpy = isinstance(data_3d, np.ndarray)

    # Convert to list for safe loop processing
    working_data = data_3d.tolist() if is_numpy else data_3d

    original_data = []

    for sample in working_data:
        new_sample = []

        for row in sample:
            new_row = row.copy()

            x_norm = row[0]

            if load_max == load_min:
                value = load_min
            else:
                value = x_norm * (load_max - load_min) + load_min

            new_row[0] = value
            new_sample.append(new_row)

        original_data.append(new_sample)

    # Return in original format
    if is_numpy:
        return np.array(original_data)
    else:
        return original_data

In [20]:
# GET THE SCALERS LIST: 
import pickle
with open("../artifacts/latest/scaler.pkl", "rb") as f:
    scaler = pickle.load(f)

SCALER_MIN = scaler.data_min_.tolist()
SCALER_MAX = scaler.data_max_.tolist()

print("SCALER_MIN =", SCALER_MIN)
print("SCALER_MAX =", SCALER_MAX)

SCALER_MIN = [1.0, -0.0885, -5.6411, 13.0, 0.23]
SCALER_MAX = [52.0, 19.8566, 16.1201, 431.0, 241.0]


In [33]:
original_inputs = minmax_inverse_load_only_3d(input_for_model, SCALER_MIN, SCALER_MAX)
original_preds  = minmax_inverse_load_only_3d(pred_onnx, SCALER_MIN, SCALER_MAX)

In [34]:
display_prediction(original_inputs, original_preds, 4, lower_limit, upper_limit, anomaly)

Number of anomalies:  147
